# NUS ST2132 — Mathematical Statistics
## Case Studies, Simulations & Interactive Bokeh Visualisations

This notebook is a **conceptual + computational companion** for NUS **ST2132 Mathematical Statistics**.

The central question of the module is:

\[
\boxed{\text{Observed data} \longrightarrow \text{Inference about an unknown population/model}}
\]

Rather than treating formulas as isolated recipes, each section follows the same structure:

1. **Concept and mathematical result**
2. **Why the result is true / how to derive it**
3. **Case study**
4. **Monte-Carlo experiment**
5. **Bokeh visualisation**
6. **Interpretation**
7. **Exercise**

### Core coverage represented here

- Random samples, statistics and sampling distributions
- Law of Large Numbers and Central Limit Theorem
- \(\chi^2\), Student-\(t\), and \(F\) sampling distributions
- Method of Moments (MOM)
- Maximum Likelihood Estimation (MLE)
- Score functions and likelihood curvature
- Fisher information
- Bias, variance and mean squared error
- Consistency and efficiency
- Sufficiency and completeness
- Cramér–Rao lower bound
- Confidence intervals
- Exact and asymptotic pivotal quantities
- Hypothesis testing
- Type-I / Type-II errors and power
- Neyman–Pearson lemma
- Likelihood-ratio tests
- Order statistics and quantiles
- QQ plots
- Chi-square goodness-of-fit and independence tests
- Distribution-free inference: sign and Wilcoxon signed-rank tests
- Optional bridge to regression inference

> **How to use this notebook:** run from top to bottom. All datasets are generated locally with fixed random seeds, making the notebook reproducible and usable offline.

## 0. Course alignment and references

The stable ST2132 core has long been described as covering random samples and statistics, method of moments, maximum likelihood, Fisher information, sufficiency and completeness, consistency and unbiasedness, sampling distributions, \(\chi^2\), \(t\), and \(F\) distributions, confidence intervals, exact/asymptotic pivotal methods, hypothesis testing, likelihood-ratio tests, and the Neyman–Pearson lemma.

Current NUS course listings identify **ST2132 — Mathematical Statistics** under the Department of Statistics & Data Science.

Useful public references:

- NUS Department of Statistics & Data Science, Courses Offered:  
  https://www.stat.nus.edu.sg/courses-offered/
- NUS-hosted historical course description with detailed ST2132 topic list:  
  https://www.comp.nus.edu.sg/~wongls/bp/courses.html
- NUS Statistics programme requirements / sample study plan:  
  https://www.stat.nus.edu.sg/wp-content/uploads/sites/8/2024/07/170624-Major-Statistics-Programme-Requirements-AY21-22-and-after.pdf
- Yap Von Bing's NUS faculty page lists ST2132 among his courses and points to companion mathematical-statistics notes:  
  https://www.stat.nus.edu.sg/faculty-members-old/yap-von-bing/

The notebook emphasises the **stable mathematical-statistics core** while adding a few useful extensions—QQ plots, nonparametric tests, categorical-data tests and regression inference—to connect theory with downstream statistical practice.

## 1. Environment setup

The notebook uses:

- **NumPy** — simulation and numerical work
- **Pandas** — tabular summaries
- **SciPy** — probability distributions and statistical tests
- **Bokeh** — interactive visualisation

No external dataset download is required.

In [1]:
import math
import warnings

import numpy as np
import pandas as pd

from scipy import optimize, stats
from scipy.special import expit

from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import (
    Band,
    ColumnDataSource,
    CustomJS,
    HoverTool,
    Slider,
    Span,
)
from bokeh.plotting import figure

warnings.filterwarnings("ignore")
output_notebook()

SEED = 2132
rng = np.random.default_rng(SEED)

pd.set_option("display.precision", 4)

print("Environment ready.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

Loading BokehJS ...

Environment ready.
NumPy: 1.26.4
Pandas: 2.3.3


# Part I — Random Samples and Sampling Distributions

## 2. Random sample, statistic and estimator

Let

\[
X_1,\ldots,X_n \overset{iid}{\sim} f(x;\theta).
\]

Before data are observed, each \(X_i\) is a random variable.

A **statistic**

\[
T=T(X_1,\ldots,X_n)
\]

is any function of the sample that does not contain an unknown parameter.

Examples:

\[
\bar X=\frac{1}{n}\sum_{i=1}^n X_i,
\qquad
S^2=\frac{1}{n-1}\sum_{i=1}^n(X_i-\bar X)^2.
\]

If \(T\) is used to estimate a parameter \(\theta\), it is called an **estimator**.

The distribution of \(T\) over repeated samples is its **sampling distribution**.

---

### Case Study 1 — Call-centre handling time

Suppose individual call handling times are positively skewed and approximately exponential:

\[
X_i\sim \operatorname{Exponential}(\text{mean}=6\text{ minutes}).
\]

A manager does not care about one random call; they care about the average handling time across a shift sample.

We will repeatedly sample calls and study the behaviour of \(\bar X\).

In [2]:
def simulate_sample_means(sampler, sample_size, repetitions=10_000):
    samples = sampler(size=(repetitions, sample_size))
    return samples.mean(axis=1)

population_mean = 6.0

sample_sizes = [1, 5, 30, 100]
sample_mean_results = {}

local_rng = np.random.default_rng(SEED)

for n in sample_sizes:
    sample_mean_results[n] = simulate_sample_means(
        lambda size: local_rng.exponential(scale=population_mean, size=size),
        sample_size=n,
        repetitions=10_000,
    )

summary = pd.DataFrame(
    {
        "n": sample_sizes,
        "Empirical E[Xbar]": [sample_mean_results[n].mean() for n in sample_sizes],
        "Empirical SD(Xbar)": [sample_mean_results[n].std(ddof=1) for n in sample_sizes],
        "Theory SD(Xbar)": [population_mean / np.sqrt(n) for n in sample_sizes],
    }
)

summary

,n,Empirical E[Xbar],Empirical SD(Xbar),Theory SD(Xbar)
0,1,6.0435,6.0332,6.0000
1,5,6.0045,2.7141,2.6833
2,30,6.0007,1.1005,1.0954
3,100,6.0028,0.5930,0.6000


In [3]:
def histogram_density_plot(values, title, x_label, bins=50):
    hist, edges = np.histogram(values, bins=bins, density=True)

    source = ColumnDataSource(
        {
            "left": edges[:-1],
            "right": edges[1:],
            "top": hist,
        }
    )

    p = figure(
        width=850,
        height=340,
        title=title,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    p.quad(
        left="left",
        right="right",
        bottom=0,
        top="top",
        source=source,
        alpha=0.55,
    )

    p.xaxis.axis_label = x_label
    p.yaxis.axis_label = "Density"

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Interval", "@left{0.00} – @right{0.00}"),
                ("Density", "@top{0.000}"),
            ]
        )
    )
    return p

plots = []

for n in sample_sizes:
    p = histogram_density_plot(
        sample_mean_results[n],
        title=f"Sampling distribution of X̄ — n={n}",
        x_label="Sample mean handling time (minutes)",
    )

    p.add_layout(
        Span(
            location=population_mean,
            dimension="height",
            line_dash="dashed",
            line_width=2,
        )
    )
    plots.append(p)

show(column(*plots))

### What the experiment illustrates

For every \(n\),

\[
E[\bar X]=\mu.
\]

But

\[
\operatorname{Var}(\bar X)=\frac{\sigma^2}{n}
\]

and therefore

\[
SE(\bar X)=\frac{\sigma}{\sqrt n}.
\]

So increasing \(n\):

- does **not** move the centre of the estimator;
- reduces its variability;
- makes estimates more precise.

This distinction—**bias versus variability**—will recur throughout ST2132.

## 3. Law of Large Numbers (LLN)

The Weak Law of Large Numbers states, under suitable conditions,

\[
\bar X_n \xrightarrow{P} \mu.
\]

For any \(\epsilon>0\),

\[
P(|\bar X_n-\mu|>\epsilon)\to 0.
\]

Interpretation:

> As the sample grows, the sample mean becomes increasingly concentrated near the population mean.

### Case Study 2 — Online transaction values

Suppose transaction amounts follow a heavy right-skewed Gamma distribution. We monitor the cumulative sample mean as more observations arrive.

In [4]:
local_rng = np.random.default_rng(SEED + 1)

n_max = 5_000
shape = 2.0
scale = 25.0
true_mean = shape * scale

transactions = local_rng.gamma(shape=shape, scale=scale, size=n_max)
running_mean = np.cumsum(transactions) / np.arange(1, n_max + 1)

source = ColumnDataSource(
    {
        "n": np.arange(1, n_max + 1),
        "running_mean": running_mean,
    }
)

p = figure(
    width=900,
    height=400,
    title="Law of Large Numbers — running transaction mean",
    x_axis_type="log",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line("n", "running_mean", source=source, line_width=2)

p.add_layout(
    Span(
        location=true_mean,
        dimension="width",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "Number of transactions n (log scale)"
p.yaxis.axis_label = "Running mean"

p.add_tools(
    HoverTool(
        tooltips=[
            ("n", "@n"),
            ("Running mean", "@running_mean{0.00}"),
        ]
    )
)

show(p)

print(f"Theoretical mean = {true_mean:.2f}")
print(f"Final running mean = {running_mean[-1]:.2f}")

Theoretical mean = 50.00
Final running mean = 50.70


## 4. Central Limit Theorem (CLT)

If \(X_i\) are IID with finite mean \(\mu\) and finite variance \(\sigma^2\), then

\[
\frac{\bar X-\mu}{\sigma/\sqrt n}
\xrightarrow{d}N(0,1).
\]

Equivalently, for sufficiently large \(n\),

\[
\bar X \approx N\left(\mu,\frac{\sigma^2}{n}\right).
\]

### Why the CLT matters

It allows us to construct approximate:

- standard errors,
- confidence intervals,
- test statistics,

even when the original population is not normal.

The next experiment deliberately uses a **strongly skewed exponential population**.

In [5]:
local_rng = np.random.default_rng(SEED + 2)

population_mean = 2.0
population_sd = 2.0
repetitions = 20_000

clt_rows = []

for n in [2, 5, 20, 50, 200]:
    means = local_rng.exponential(
        scale=population_mean,
        size=(repetitions, n)
    ).mean(axis=1)

    z = (means - population_mean) / (population_sd / np.sqrt(n))

    clt_rows.append(
        {
            "n": n,
            "mean(z)": z.mean(),
            "sd(z)": z.std(ddof=1),
            "skew(z)": stats.skew(z),
            "P(|Z| <= 1.96)": np.mean(np.abs(z) <= 1.96),
        }
    )

pd.DataFrame(clt_rows)

,n,mean(z),sd(z),skew(z),P(|Z| <= 1.96)
0,2,0.0069,1.0115,1.4093,0.9484
1,5,0.0161,1.0108,0.9205,0.9535
2,20,-0.0057,1.0022,0.4562,0.9522
3,50,0.0018,0.9995,0.2574,0.9507
4,200,0.0036,0.9987,0.1416,0.9504


In [6]:
x_grid = np.linspace(-4, 4, 600)
normal_density = stats.norm.pdf(x_grid)

clt_plots = []
local_rng = np.random.default_rng(SEED + 3)

for n in [2, 20, 200]:
    means = local_rng.exponential(
        scale=population_mean,
        size=(12_000, n)
    ).mean(axis=1)

    z = (means - population_mean) / (population_sd / np.sqrt(n))

    p = histogram_density_plot(
        z,
        title=f"Standardised sample mean versus N(0,1) — n={n}",
        x_label="Z = (X̄ - μ)/(σ/√n)",
        bins=55,
    )

    p.line(
        x_grid,
        normal_density,
        line_width=3,
        legend_label="Standard Normal density",
    )

    p.legend.click_policy = "hide"
    clt_plots.append(p)

show(column(*clt_plots))

### Exercise 1

Replace the exponential population with:

1. Uniform\((0,1)\)
2. Bernoulli\((0.1)\)
3. Lognormal with a large variance

For each population, estimate how large \(n\) must be before the standardised sample mean appears approximately Gaussian.

<details>
<summary><b>Guidance / expected insight</b></summary>

There is no universal “\(n\ge 30\)” CLT rule.

The required sample size depends strongly on:

- skewness,
- tail heaviness,
- discreteness,
- rarity of events.

A highly skewed or rare-event Bernoulli population can need substantially larger \(n\).
</details>

# Part II — Exact Sampling Distributions

## 5. \(\chi^2\), Student-\(t\), and \(F\)

For a normal sample

\[
X_1,\ldots,X_n\sim N(\mu,\sigma^2),
\]

three exact results dominate classical inference.

### Sample variance

\[
\frac{(n-1)S^2}{\sigma^2}
\sim \chi^2_{n-1}.
\]

### Studentised sample mean

\[
T=
\frac{\bar X-\mu}{S/\sqrt n}
\sim t_{n-1}.
\]

### Ratio of independent sample variances

For two independent normal samples,

\[
\frac{S_1^2/\sigma_1^2}{S_2^2/\sigma_2^2}
\sim F_{n_1-1,n_2-1}.
\]

These results allow **exact finite-sample inference**, not merely CLT approximations.

In [7]:
x_chi = np.linspace(0.001, 30, 700)
x_t = np.linspace(-5, 5, 700)
x_f = np.linspace(0.001, 5, 700)

p_chi = figure(
    width=850,
    height=330,
    title="Chi-square distributions",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
for df in [1, 2, 5, 10, 20]:
    p_chi.line(
        x_chi,
        stats.chi2.pdf(x_chi, df),
        line_width=2,
        legend_label=f"df={df}",
    )
p_chi.legend.click_policy = "hide"
p_chi.xaxis.axis_label = "x"
p_chi.yaxis.axis_label = "Density"

p_t = figure(
    width=850,
    height=330,
    title="Student-t approaches Normal as degrees of freedom increase",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p_t.line(
    x_t,
    stats.norm.pdf(x_t),
    line_width=3,
    legend_label="N(0,1)",
)
for df in [1, 3, 10, 30]:
    p_t.line(
        x_t,
        stats.t.pdf(x_t, df),
        line_width=2,
        legend_label=f"t(df={df})",
    )
p_t.legend.click_policy = "hide"
p_t.xaxis.axis_label = "x"
p_t.yaxis.axis_label = "Density"

p_f = figure(
    width=850,
    height=330,
    title="F distributions",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
for df1, df2 in [(2, 5), (5, 10), (10, 10), (30, 30)]:
    p_f.line(
        x_f,
        stats.f.pdf(x_f, df1, df2),
        line_width=2,
        legend_label=f"F({df1},{df2})",
    )
p_f.legend.click_policy = "hide"
p_f.xaxis.axis_label = "x"
p_f.yaxis.axis_label = "Density"

show(column(p_chi, p_t, p_f))

### Case Study 3 — Manufacturing variance

A bottling process is designed to have fill standard deviation \(\sigma=2\) ml.

If the population is approximately normal and we sample \(n=20\) bottles, the randomness of the observed sample variance is governed exactly by

\[
\frac{19S^2}{\sigma^2}\sim\chi^2_{19}.
\]

Let's verify the result by simulation.

In [8]:
local_rng = np.random.default_rng(SEED + 4)

mu = 500.0
sigma = 2.0
n = 20
repetitions = 30_000

samples = local_rng.normal(mu, sigma, size=(repetitions, n))
sample_vars = samples.var(axis=1, ddof=1)

chi_stat = (n - 1) * sample_vars / sigma**2

hist, edges = np.histogram(chi_stat, bins=60, density=True)

p = histogram_density_plot(
    chi_stat,
    title="Simulation check: (n−1)S²/σ² versus χ²(n−1)",
    x_label="Scaled sample variance",
    bins=60,
)

grid = np.linspace(0.01, np.quantile(chi_stat, 0.997), 600)
p.line(
    grid,
    stats.chi2.pdf(grid, df=n - 1),
    line_width=3,
    legend_label=f"χ²({n - 1}) theoretical density",
)
p.legend.click_policy = "hide"

show(p)

# Part III — Point Estimation

## 6. Method of Moments (MOM)

Let the first theoretical moment be

\[
E_\theta[X]=m_1(\theta).
\]

The corresponding sample moment is

\[
\bar X.
\]

MOM solves

\[
m_1(\theta)=\bar X.
\]

With \(k\) unknown parameters, one commonly equates \(k\) theoretical moments with \(k\) empirical moments.

---

### Case Study 4 — Server failure intervals

Suppose time between server failures follows

\[
X_i\sim \operatorname{Exponential}(\lambda),
\qquad
f(x;\lambda)=\lambda e^{-\lambda x}.
\]

Since

\[
E[X]=\frac{1}{\lambda},
\]

MOM gives

\[
\hat\lambda_{\text{MOM}}=\frac{1}{\bar X}.
\]

In [9]:
local_rng = np.random.default_rng(SEED + 5)

true_lambda = 0.25
n = 80

failure_intervals = local_rng.exponential(
    scale=1 / true_lambda,
    size=n
)

lambda_mom = 1 / failure_intervals.mean()

print(f"True λ       : {true_lambda:.4f}")
print(f"MOM estimate : {lambda_mom:.4f}")
print(f"Sample mean  : {failure_intervals.mean():.4f}")

True λ       : 0.2500
MOM estimate : 0.2623
Sample mean  : 3.8129


## 7. Maximum Likelihood Estimation (MLE)

For observed sample \(x=(x_1,\ldots,x_n)\),

\[
L(\theta;x)=\prod_{i=1}^n f(x_i;\theta).
\]

The MLE is

\[
\hat\theta_{\text{MLE}}
=
\arg\max_\theta L(\theta;x).
\]

It is almost always numerically preferable to maximise the **log-likelihood**

\[
\ell(\theta)
=
\sum_{i=1}^n\log f(x_i;\theta).
\]

For the exponential model,

\[
\ell(\lambda)
=
n\log\lambda-\lambda\sum_i x_i.
\]

Differentiate:

\[
\frac{\partial\ell}{\partial\lambda}
=
\frac{n}{\lambda}-\sum_i x_i.
\]

Setting the score to zero gives

\[
\hat\lambda_{\text{MLE}}
=
\frac{n}{\sum_i x_i}
=
\frac{1}{\bar X}.
\]

So in this model, MOM and MLE happen to coincide.

In [10]:
lambda_grid = np.linspace(0.05, 0.60, 600)

log_likelihood = (
    n * np.log(lambda_grid)
    - lambda_grid * failure_intervals.sum()
)

lambda_mle = lambda_grid[np.argmax(log_likelihood)]

source = ColumnDataSource(
    {
        "lambda": lambda_grid,
        "log_likelihood": log_likelihood,
    }
)

p = figure(
    width=900,
    height=400,
    title="Exponential log-likelihood",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    "lambda",
    "log_likelihood",
    source=source,
    line_width=3,
)

p.add_layout(
    Span(
        location=lambda_mom,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "Candidate λ"
p.yaxis.axis_label = "Log-likelihood"

p.add_tools(
    HoverTool(
        tooltips=[
            ("λ", "@lambda{0.0000}"),
            ("log L", "@log_likelihood{0.00}"),
        ]
    )
)

show(p)

print(f"Grid MLE ≈ {lambda_mle:.4f}")
print(f"Analytic MLE = {lambda_mom:.4f}")

Grid MLE ≈ 0.2621
Analytic MLE = 0.2623


### ML connection

If

\[
Y_i\sim N(\mu_i,\sigma^2),
\]

then maximising Gaussian likelihood with fixed \(\sigma\) is equivalent to minimising

\[
\sum_i (y_i-\mu_i)^2.
\]

Thus ordinary squared-error fitting is a likelihood procedure under a Gaussian-noise assumption.

Likewise, logistic regression maximises a Bernoulli likelihood.

MLE is therefore not merely a classical-statistics technique—it is one of the mathematical foundations of machine learning.

## 8. Score function and likelihood curvature

The **score** is

\[
U(\theta)
=
\frac{\partial}{\partial\theta}\ell(\theta).
\]

An interior MLE usually satisfies

\[
U(\hat\theta)=0.
\]

The second derivative measures local curvature:

\[
\ell''(\theta).
\]

A sharply curved log-likelihood means the data strongly localise \(\theta\); a flat likelihood means substantial uncertainty.

In [11]:
local_rng = np.random.default_rng(SEED + 6)

true_mu = 5.0
sigma = 2.0
sample_sizes = [5, 20, 100, 500]
mu_grid = np.linspace(3.5, 6.5, 600)

p = figure(
    width=900,
    height=430,
    title="Likelihood concentration as sample size increases",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for n in sample_sizes:
    x = local_rng.normal(true_mu, sigma, size=n)
    ll = np.array([
        stats.norm.logpdf(x, loc=mu, scale=sigma).sum()
        for mu in mu_grid
    ])

    # Shift each curve so all maxima equal 0; shape is what matters.
    ll = ll - ll.max()

    p.line(
        mu_grid,
        ll,
        line_width=2,
        legend_label=f"n={n}",
    )

p.add_layout(
    Span(
        location=true_mu,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "Candidate μ"
p.yaxis.axis_label = "Relative log-likelihood"
p.legend.click_policy = "hide"

show(p)

# Part IV — Evaluating Estimators

## 9. Bias, variance and mean squared error

For an estimator \(\hat\theta\),

\[
\operatorname{Bias}(\hat\theta)
=
E[\hat\theta]-\theta.
\]

Its mean squared error is

\[
MSE(\hat\theta)
=
E[(\hat\theta-\theta)^2].
\]

The key decomposition is

\[
\boxed{
MSE(\hat\theta)
=
\operatorname{Var}(\hat\theta)
+
\operatorname{Bias}(\hat\theta)^2
}
\]

This is the classical ancestor of the bias–variance trade-off in predictive modelling.

---

### Case Study 5 — Estimating a Bernoulli conversion rate

Suppose

\[
X_i\sim Bernoulli(p).
\]

Compare:

\[
\hat p_1=\bar X
\]

with a shrinkage estimator

\[
\hat p_2=\frac{X+1}{n+2},
\]

where \(X=\sum_iX_i\).

\(\hat p_2\) is biased but may have lower MSE near certain values of \(p\).

In [12]:
def estimator_risk_experiment(p_true, n=10, repetitions=100_000, seed=SEED):
    local_rng = np.random.default_rng(seed)

    successes = local_rng.binomial(
        n=n,
        p=p_true,
        size=repetitions
    )

    mle = successes / n
    shrink = (successes + 1) / (n + 2)

    def summarize(est):
        bias = est.mean() - p_true
        variance = est.var(ddof=0)
        mse = np.mean((est - p_true) ** 2)

        return bias, variance, mse

    return summarize(mle), summarize(shrink)

rows = []

for p_true in [0.05, 0.15, 0.30, 0.50, 0.80, 0.95]:
    mle_stats, shrink_stats = estimator_risk_experiment(p_true)

    rows.extend(
        [
            {
                "p": p_true,
                "Estimator": "X/n",
                "Bias": mle_stats[0],
                "Variance": mle_stats[1],
                "MSE": mle_stats[2],
            },
            {
                "p": p_true,
                "Estimator": "(X+1)/(n+2)",
                "Bias": shrink_stats[0],
                "Variance": shrink_stats[1],
                "MSE": shrink_stats[2],
            },
        ]
    )

risk_table = pd.DataFrame(rows)
risk_table

,p,Estimator,Bias,Variance,MSE
0,0.05,X/n,-8.9000e-05,0.0048,0.0048
1,0.05,(X+1)/(n+2),7.4926e-02,0.0033,0.0089
2,0.15,X/n,-6.5000e-05,0.0127,0.0127
3,0.15,(X+1)/(n+2),5.8279e-02,0.0088,0.0122
4,0.30,X/n,6.5000e-05,0.0209,0.0209
5,0.30,(X+1)/(n+2),3.3388e-02,0.0145,0.0156
6,0.50,X/n,1.1500e-04,0.0249,0.0249
7,0.50,(X+1)/(n+2),9.5833e-05,0.0173,0.0173
8,0.80,X/n,2.1000e-05,0.0160,0.0160
9,0.80,(X+1)/(n+2),-4.9983e-02,0.0111,0.0136


In [13]:
p_grid = np.linspace(0.01, 0.99, 120)

mse_mle = []
mse_shrink = []

for i, p_true in enumerate(p_grid):
    local_rng = np.random.default_rng(SEED + i)

    successes = local_rng.binomial(
        n=10,
        p=p_true,
        size=25_000
    )

    mle = successes / 10
    shrink = (successes + 1) / 12

    mse_mle.append(np.mean((mle - p_true) ** 2))
    mse_shrink.append(np.mean((shrink - p_true) ** 2))

p = figure(
    width=900,
    height=420,
    title="Estimator risk: unbiased MLE versus biased shrinkage estimator",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    p_grid,
    mse_mle,
    line_width=3,
    legend_label="MLE: X/n",
)

p.line(
    p_grid,
    mse_shrink,
    line_width=3,
    line_dash="dashed",
    legend_label="Shrinkage: (X+1)/(n+2)",
)

p.xaxis.axis_label = "True Bernoulli probability p"
p.yaxis.axis_label = "Monte-Carlo MSE"
p.legend.click_policy = "hide"

show(p)

### Interpretation

Unbiasedness is desirable, but it is not synonymous with optimality.

An estimator can trade a small amount of bias for a substantial reduction in variance.

That is exactly why

\[
MSE = Variance + Bias^2
\]

is often a more useful measure of estimator quality than bias alone.

## 10. Consistency

An estimator sequence \(\hat\theta_n\) is **consistent** if

\[
\hat\theta_n\xrightarrow{P}\theta.
\]

For every \(\epsilon>0\),

\[
P(|\hat\theta_n-\theta|>\epsilon)\to0.
\]

For the sample mean, LLN supplies consistency.

We can visualise this by asking:

> What fraction of repeated estimates fall more than \(\epsilon\) away from the truth?

In [14]:
local_rng = np.random.default_rng(SEED + 20)

theta = 3.0
sigma = 4.0
epsilon = 0.5

n_values = np.unique(
    np.round(np.logspace(0.5, 3.2, 45)).astype(int)
)

failure_prob = []

for n in n_values:
    means = local_rng.normal(
        theta,
        sigma,
        size=(8_000, n)
    ).mean(axis=1)

    failure_prob.append(
        np.mean(np.abs(means - theta) > epsilon)
    )

p = figure(
    width=900,
    height=400,
    title="Consistency: P(|X̄ − μ| > ε) shrinks with n",
    x_axis_type="log",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    n_values,
    failure_prob,
    line_width=3,
)

p.scatter(
    n_values,
    failure_prob,
    size=6,
)

p.xaxis.axis_label = "Sample size n (log scale)"
p.yaxis.axis_label = f"P(|X̄ − μ| > {epsilon})"

show(p)

## 11. Fisher information

For one observation with density \(f(x;\theta)\),

\[
I(\theta)
=
E_\theta
\left[
\left(
\frac{\partial}{\partial\theta}
\log f(X;\theta)
\right)^2
\right].
\]

Under regularity conditions,

\[
I(\theta)
=
-E_\theta
\left[
\frac{\partial^2}{\partial\theta^2}
\log f(X;\theta)
\right].
\]

For IID observations,

\[
I_n(\theta)=nI_1(\theta).
\]

So information grows linearly with sample size.

For a normal mean with known variance,

\[
I_n(\mu)=\frac{n}{\sigma^2}.
\]

The asymptotic MLE variance is approximately

\[
\operatorname{Var}(\hat\mu)
\approx
\frac{1}{I_n(\mu)}
=
\frac{\sigma^2}{n}.
\]

## 12. Cramér–Rao Lower Bound (CRLB)

For an unbiased estimator under regularity conditions,

\[
\operatorname{Var}(\hat\theta)
\ge
\frac{1}{I_n(\theta)}.
\]

This places a fundamental lower bound on variance.

### Bernoulli example

For \(X_i\sim Bernoulli(p)\),

\[
I_n(p)=\frac{n}{p(1-p)}.
\]

Therefore

\[
\frac{1}{I_n(p)}
=
\frac{p(1-p)}{n}.
\]

But

\[
\operatorname{Var}(\bar X)
=
\frac{p(1-p)}{n}.
\]

Hence \(\bar X\) reaches the CRLB and is efficient in this setting.

In [15]:
p_grid = np.linspace(0.01, 0.99, 500)

for n in [10, 50, 200]:
    crlb = p_grid * (1 - p_grid) / n

    p_fig = figure(
        width=850,
        height=300,
        title=f"Bernoulli CRLB for p — n={n}",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    p_fig.line(
        p_grid,
        crlb,
        line_width=3,
    )

    p_fig.xaxis.axis_label = "p"
    p_fig.yaxis.axis_label = "Minimum unbiased variance"

    show(p_fig)

# Part V — Sufficiency and Completeness

## 13. Sufficiency

A statistic \(T(X)\) is sufficient for \(\theta\) if the conditional distribution of the full data given \(T\) does not depend on \(\theta\).

The **Neyman–Fisher factorisation theorem** gives a practical criterion.

If the joint density can be written as

\[
f(x_1,\ldots,x_n;\theta)
=
g(T(x),\theta)h(x),
\]

then \(T\) is sufficient for \(\theta\).

### Bernoulli example

For

\[
X_i\sim Bernoulli(p),
\]

\[
L(p)
=
\prod_i p^{x_i}(1-p)^{1-x_i}
=
p^{\sum_i x_i}(1-p)^{n-\sum_i x_i}.
\]

The data enter the likelihood only through

\[
T=\sum_iX_i.
\]

So \(T\) is sufficient for \(p\).

### Information-compression interpretation

For inference about \(p\),

\[
(x_1,\ldots,x_n)
\quad\longrightarrow\quad
\sum_i x_i
\]

loses no likelihood information.

In [38]:
n = 8
success_count = 3

sequences = [
    np.array([1, 1, 1, 0, 0, 0, 0, 0]),
    np.array([1, 0, 1, 0, 1, 0, 0, 0]),
    np.array([0, 0, 1, 1, 0, 0, 1, 0]),
]

p_grid = np.linspace(0.01, 0.99, 500)

fig = figure(
    width=900,
    height=400,
    title="Different Bernoulli sequences with the same sufficient statistic",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for idx, seq in enumerate(sequences, start=1):
    k = seq.sum()

    likelihood = (
        p_grid**k *
        (1 - p_grid)**(n - k)
    )

    likelihood = likelihood / likelihood.max()

    fig.line(
        p_grid,
        likelihood,
        line_width=2,
        legend_label=f"Sequence {idx}: sum={k}",
    )

fig.xaxis.axis_label = "p"
fig.yaxis.axis_label = "Relative likelihood"
fig.legend.click_policy = "hide"

show(fig)

All three curves coincide because the likelihood depends on the observations only through the count of successes.

That is sufficiency made visible.

---

## 14. Completeness

A statistic \(T\) is complete if

\[
E_\theta[g(T)]=0
\quad\text{for every }\theta
\]

implies

\[
P_\theta(g(T)=0)=1.
\]

Completeness is mainly a uniqueness property.

### Why sufficiency + completeness matter

The **Lehmann–Scheffé theorem** states that if \(T\) is complete and sufficient and an unbiased estimator is a function of \(T\), then that estimator is the unique minimum-variance unbiased estimator (UMVU).

This is a theoretical result rather than a visual one, but it gives an important design principle:

\[
\boxed{
\text{Find a complete sufficient statistic}
\rightarrow
\text{build an unbiased function of it}
\rightarrow
\text{obtain the UMVU estimator}
}
\]

# Part VI — Confidence Intervals and Pivotal Quantities

## 15. Pivotal quantities

A pivotal quantity

\[
Q(X,\theta)
\]

has a distribution that does **not depend on unknown parameters**.

Example, if \(\sigma\) is known:

\[
Z=
\frac{\bar X-\mu}{\sigma/\sqrt n}
\sim N(0,1).
\]

This can be inverted:

\[
P\left(
-z_{\alpha/2}
\le
\frac{\bar X-\mu}{\sigma/\sqrt n}
\le
z_{\alpha/2}
\right)
=
1-\alpha.
\]

Therefore,

\[
\boxed{
\mu\in
\left[
\bar X-z_{\alpha/2}\frac{\sigma}{\sqrt n},
\;
\bar X+z_{\alpha/2}\frac{\sigma}{\sqrt n}
\right]
}
\]

with confidence level \(1-\alpha\).

### Case Study 6 — Bottle fill-weight confidence intervals

A production line fills bottles with mean volume \(\mu\). Assume a known process standard deviation of 2 ml.

We repeatedly sample 30 bottles and construct 95% confidence intervals.

If the procedure is correctly calibrated, approximately 95% of intervals should contain the fixed true mean.

In [17]:
local_rng = np.random.default_rng(SEED + 30)

mu = 500.0
sigma = 2.0
n = 30
confidence = 0.95
alpha = 1 - confidence
zcrit = stats.norm.ppf(1 - alpha / 2)

repetitions = 100

samples = local_rng.normal(
    mu,
    sigma,
    size=(repetitions, n)
)

means = samples.mean(axis=1)
margin = zcrit * sigma / np.sqrt(n)

lower = means - margin
upper = means + margin
covered = (lower <= mu) & (mu <= upper)

source = ColumnDataSource(
    {
        "experiment": np.arange(repetitions),
        "lower": lower,
        "upper": upper,
        "mean": means,
        "covered": covered.astype(str),
    }
)

p = figure(
    width=950,
    height=600,
    title="Repeated 95% confidence intervals for μ",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for i in range(repetitions):
    p.segment(
        x0=[lower[i]],
        y0=[i],
        x1=[upper[i]],
        y1=[i],
        line_width=2,
        line_dash="solid" if covered[i] else "dashed",
    )

p.add_layout(
    Span(
        location=mu,
        dimension="height",
        line_dash="dashed",
        line_width=3,
    )
)

p.xaxis.axis_label = "Candidate mean fill volume (ml)"
p.yaxis.axis_label = "Repeated experiment"

show(p)

print(f"Empirical coverage = {covered.mean():.3f}")

Empirical coverage = 0.940


### Frequentist interpretation

A common incorrect statement is:

> “There is a 95% probability that \(\mu\) lies inside this realised interval.”

In classical frequentist inference, \(\mu\) is fixed. The random object is the interval-generating procedure.

Correct interpretation:

> In repeated sampling, approximately 95% of intervals produced by this procedure cover \(\mu\).

---

## 16. Unknown \(\sigma\): Student-\(t\) confidence interval

When the normal population variance is unknown,

\[
T
=
\frac{\bar X-\mu}{S/\sqrt n}
\sim t_{n-1}.
\]

Therefore,

\[
\boxed{
\bar X
\pm
t_{n-1,1-\alpha/2}
\frac{S}{\sqrt n}
}
\]

is an exact normal-theory confidence interval.

In [18]:
local_rng = np.random.default_rng(SEED + 31)

mu = 12
sigma = 5
n = 8
repetitions = 25_000
alpha = 0.05

samples = local_rng.normal(mu, sigma, size=(repetitions, n))
means = samples.mean(axis=1)
sds = samples.std(axis=1, ddof=1)

tcrit = stats.t.ppf(1 - alpha/2, df=n - 1)
zcrit = stats.norm.ppf(1 - alpha/2)

t_lower = means - tcrit * sds / np.sqrt(n)
t_upper = means + tcrit * sds / np.sqrt(n)

z_lower = means - zcrit * sds / np.sqrt(n)
z_upper = means + zcrit * sds / np.sqrt(n)

comparison = pd.DataFrame(
    {
        "Procedure": [
            "Correct t interval",
            "Naive normal interval using S",
        ],
        "Empirical coverage": [
            np.mean((t_lower <= mu) & (mu <= t_upper)),
            np.mean((z_lower <= mu) & (mu <= z_upper)),
        ],
        "Mean width": [
            np.mean(t_upper - t_lower),
            np.mean(z_upper - z_lower),
        ],
    }
)

comparison

,Procedure,Empirical coverage,Mean width
0,Correct t interval,0.9494,8.0940
1,Naive normal interval using S,0.9076,6.7089


This experiment shows **why Student's \(t\)** exists.

When \(\sigma\) is estimated using \(S\), the denominator adds uncertainty. The \(t\) distribution compensates with heavier tails.

# Part VII — Hypothesis Testing

## 17. Testing framework

A hypothesis test separates the parameter space into:

\[
H_0:\theta\in\Theta_0
\]

versus

\[
H_1:\theta\in\Theta_1.
\]

A test statistic \(T(X)\) maps the observed sample into a decision rule.

Two errors are possible:

\[
\alpha
=
P(\text{reject }H_0\mid H_0\text{ true})
\]

and

\[
\beta(\theta)
=
P_\theta(\text{fail to reject }H_0\mid H_1\text{ true}).
\]

The **power function** is

\[
\pi(\theta)
=
P_\theta(\text{reject }H_0)
=
1-\beta(\theta)
\]

for alternatives.

### Case Study 7 — A/B conversion uplift

A website's existing conversion probability is

\[
p_0=0.10.
\]

A redesigned experience is tested on \(n=500\) users.

We test

\[
H_0:p=0.10
\]

against

\[
H_1:p>0.10.
\]

Using a normal approximation,

\[
Z=
\frac{\hat p-p_0}
{\sqrt{p_0(1-p_0)/n}}.
\]

Reject when

\[
Z>z_{1-\alpha}.
\]

In [19]:
p0 = 0.10
n = 500
alpha = 0.05

zcrit = stats.norm.ppf(1 - alpha)
p_grid = np.linspace(0.04, 0.22, 500)

# Rejection threshold expressed in terms of p-hat.
p_hat_critical = p0 + zcrit * np.sqrt(p0 * (1 - p0) / n)

# Approximate power under true p.
power = 1 - stats.norm.cdf(
    (p_hat_critical - p_grid)
    / np.sqrt(p_grid * (1 - p_grid) / n)
)

p = figure(
    width=900,
    height=420,
    title="Power function for one-sided conversion-rate test",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    p_grid,
    power,
    line_width=3,
)

p.add_layout(
    Span(
        location=p0,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "True conversion probability p"
p.yaxis.axis_label = "P(reject H₀ | p)"

show(p)

print(f"Approximate critical p-hat = {p_hat_critical:.4f}")
print(f"Power at p=p0 ≈ {np.interp(p0, p_grid, power):.4f}")
print(f"Power at p=0.12 ≈ {np.interp(0.12, p_grid, power):.4f}")
print(f"Power at p=0.15 ≈ {np.interp(0.15, p_grid, power):.4f}")

Approximate critical p-hat = 0.1221
Power at p=p0 ≈ 0.0500
Power at p=0.12 ≈ 0.4434
Power at p=0.15 ≈ 0.9599


### What controls power?

Power generally increases when:

- the true effect moves farther from the null;
- sample size \(n\) increases;
- measurement noise decreases;
- significance level \(\alpha\) increases.

The last point exposes a trade-off:

\[
\alpha\downarrow
\quad\Rightarrow\quad
\text{harder to reject }H_0
\quad\Rightarrow\quad
\text{power usually decreases}.
\]

In [20]:
sample_sizes = [100, 250, 500, 1_000, 5_000]

p = figure(
    width=900,
    height=430,
    title="Power versus sample size in the A/B conversion test",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for n_ in sample_sizes:
    threshold = p0 + zcrit * np.sqrt(p0 * (1 - p0) / n_)

    power_n = 1 - stats.norm.cdf(
        (threshold - p_grid)
        / np.sqrt(p_grid * (1 - p_grid) / n_)
    )

    p.line(
        p_grid,
        power_n,
        line_width=2,
        legend_label=f"n={n_}",
    )

p.add_layout(
    Span(
        location=p0,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "True conversion probability p"
p.yaxis.axis_label = "Power"
p.legend.click_policy = "hide"

show(p)

## 18. p-values

A p-value is

\[
P_{H_0}
(
\text{test statistic at least as incompatible with }H_0
\text{ as the observed statistic}
).
\]

It is **not**

\[
P(H_0\mid\text{data}).
\]

Those are very different probabilities.

### Small simulation: p-values under \(H_0\)

For a continuous correctly calibrated test, p-values generated under the null should be approximately Uniform\((0,1)\).

In [21]:
local_rng = np.random.default_rng(SEED + 40)

n = 40
repetitions = 25_000
mu0 = 0.0
sigma = 1.0

samples = local_rng.normal(
    mu0,
    sigma,
    size=(repetitions, n)
)

z_stats = samples.mean(axis=1) / (sigma / np.sqrt(n))
p_values = 2 * stats.norm.sf(np.abs(z_stats))

p = histogram_density_plot(
    p_values,
    title="Distribution of p-values when H₀ is true",
    x_label="p-value",
    bins=40,
)

p.line(
    [0, 1],
    [1, 1],
    line_width=3,
    line_dash="dashed",
    legend_label="Uniform(0,1) density",
)

p.legend.click_policy = "hide"

show(p)

print(f"Fraction p < 0.05 = {np.mean(p_values < 0.05):.4f}")

Fraction p < 0.05 = 0.0528


# Part VIII — Neyman–Pearson and Likelihood-Ratio Testing

## 19. Neyman–Pearson lemma

For a **simple null**

\[
H_0:\theta=\theta_0
\]

against a **simple alternative**

\[
H_1:\theta=\theta_1,
\]

the most powerful size-\(\alpha\) test rejects \(H_0\) for sufficiently large values of

\[
\frac{L(\theta_1;x)}{L(\theta_0;x)}
\]

or equivalently sufficiently small values of the reverse likelihood ratio.

The result says something profound:

> If you must spend a fixed Type-I error budget \(\alpha\), allocate rejection probability to observations that are most characteristic of \(H_1\) relative to \(H_0\).

---

### Case Study 8 — Fraud-score distribution shift

Suppose a fraud score \(X\) has

\[
H_0:X\sim N(0,1)
\]

for legitimate transactions and

\[
H_1:X\sim N(1.5,1)
\]

for fraudulent transactions.

The likelihood ratio is monotone in \(X\), so Neyman–Pearson implies that the optimal test rejects for large scores.

In [22]:
mu0 = 0.0
mu1 = 1.5
sigma = 1.0
alpha = 0.05

critical_x = stats.norm.ppf(1 - alpha, loc=mu0, scale=sigma)

x_grid = np.linspace(-4, 5, 800)
f0 = stats.norm.pdf(x_grid, mu0, sigma)
f1 = stats.norm.pdf(x_grid, mu1, sigma)
lr = f1 / np.maximum(f0, 1e-300)

p1 = figure(
    width=900,
    height=370,
    title="Neyman–Pearson case study: null and alternative densities",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p1.line(
    x_grid,
    f0,
    line_width=3,
    legend_label="H₀ density",
)
p1.line(
    x_grid,
    f1,
    line_width=3,
    line_dash="dashed",
    legend_label="H₁ density",
)

p1.add_layout(
    Span(
        location=critical_x,
        dimension="height",
        line_dash="dotted",
        line_width=3,
    )
)

p1.xaxis.axis_label = "Fraud score"
p1.yaxis.axis_label = "Density"
p1.legend.click_policy = "hide"

p2 = figure(
    width=900,
    height=330,
    title="Likelihood ratio f₁(x)/f₀(x)",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p2.line(
    x_grid,
    lr,
    line_width=3,
)

p2.add_layout(
    Span(
        location=critical_x,
        dimension="height",
        line_dash="dotted",
        line_width=3,
    )
)

p2.xaxis.axis_label = "Fraud score"
p2.yaxis.axis_label = "Likelihood ratio"

show(column(p1, p2))

power_np = 1 - stats.norm.cdf(
    critical_x,
    loc=mu1,
    scale=sigma
)

print(f"Critical threshold = {critical_x:.3f}")
print(f"Type-I error = {alpha:.3f}")
print(f"Power against μ={mu1} = {power_np:.3f}")

Critical threshold = 1.645
Type-I error = 0.050
Power against μ=1.5 = 0.442


## 20. Generalised Likelihood-Ratio Test (LRT)

For composite hypotheses,

\[
\Lambda(x)
=
\frac{
\sup_{\theta\in\Theta_0}L(\theta;x)
}{
\sup_{\theta\in\Theta}L(\theta;x)
}.
\]

Small \(\Lambda\) indicates that restricting the model to \(H_0\) substantially worsens the best achievable likelihood.

Often we use

\[
-2\log\Lambda.
\]

Under regularity conditions and large samples,

\[
-2\log\Lambda
\xrightarrow{d}
\chi^2_k,
\]

where \(k\) is the difference in dimensionality between unrestricted and null parameter spaces.

This is **Wilks' theorem**.

### Case Study 9 — Testing a normal mean by likelihood ratio

Suppose \(X_i\sim N(\mu,\sigma^2)\) with known \(\sigma\).

Test

\[
H_0:\mu=0
\]

against

\[
H_1:\mu\ne0.
\]

The unrestricted MLE is

\[
\hat\mu=\bar X.
\]

The LRT statistic is

\[
-2\log\Lambda
=
\frac{n\bar X^2}{\sigma^2},
\]

which under \(H_0\) follows

\[
\chi^2_1.
\]

Thus the familiar normal mean test can be recovered from likelihood-ratio theory.

In [23]:
local_rng = np.random.default_rng(SEED + 50)

n = 25
sigma = 2.0
mu0 = 0.0
repetitions = 25_000

samples = local_rng.normal(
    mu0,
    sigma,
    size=(repetitions, n)
)

sample_means = samples.mean(axis=1)

lrt_stat = n * sample_means**2 / sigma**2

p = histogram_density_plot(
    lrt_stat,
    title="LRT statistic under H₀ versus χ²(1)",
    x_label="-2 log Λ",
    bins=60,
)

grid = np.linspace(
    0.001,
    np.quantile(lrt_stat, 0.998),
    700,
)

p.line(
    grid,
    stats.chi2.pdf(grid, df=1),
    line_width=3,
    legend_label="χ²(1)",
)

p.legend.click_policy = "hide"

show(p)

critical = stats.chi2.ppf(0.95, df=1)
print(f"Empirical rejection rate at 5% = {np.mean(lrt_stat > critical):.4f}")

Empirical rejection rate at 5% = 0.0506


# Part IX — Order Statistics and Distribution Diagnostics

## 21. Order statistics

Sort an IID sample:

\[
X_{(1)}
\le
X_{(2)}
\le
\cdots
\le
X_{(n)}.
\]

Then:

- \(X_{(1)}\) is the sample minimum;
- \(X_{(n)}\) is the sample maximum;
- middle order statistics determine empirical quantiles and medians.

For a continuous CDF \(F\) and density \(f\),

\[
f_{X_{(k)}}(x)
=
\frac{n!}{(k-1)!(n-k)!}
[F(x)]^{k-1}
[1-F(x)]^{n-k}
f(x).
\]

A famous special case:

If

\[
X_i\sim Uniform(0,1),
\]

then

\[
X_{(k)}
\sim Beta(k,n+1-k).
\]

In [24]:
local_rng = np.random.default_rng(SEED + 60)

n = 12
k = 4
repetitions = 30_000

samples = local_rng.uniform(
    0,
    1,
    size=(repetitions, n)
)

order_stat = np.sort(samples, axis=1)[:, k - 1]

p = histogram_density_plot(
    order_stat,
    title=f"Distribution of the {k}th order statistic from n={n} Uniform(0,1) draws",
    x_label=f"X_({k})",
    bins=55,
)

grid = np.linspace(0.001, 0.999, 700)
p.line(
    grid,
    stats.beta.pdf(grid, k, n + 1 - k),
    line_width=3,
    legend_label=f"Beta({k}, {n + 1 - k})",
)

p.legend.click_policy = "hide"

show(p)

## 22. QQ plots

A quantile-quantile plot compares sample order statistics against theoretical quantiles.

For sorted observations \(x_{(i)}\), choose plotting probabilities such as

\[
p_i=\frac{i-0.5}{n},
\]

then calculate theoretical quantiles

\[
q_i=F^{-1}(p_i).
\]

Plot

\[
(q_i,x_{(i)}).
\]

If the assumed family fits well, points should be approximately linear.

### Case Study 10 — API latency tails

A normal model can fit the centre of a latency distribution while badly missing the tails.

We compare:

1. genuine normal data;
2. heavy-tailed \(t_3\) data.

Both are standardised before creating normal QQ plots.

In [25]:
def qq_data(sample, dist=stats.norm):
    x = np.sort(np.asarray(sample))
    n = len(x)

    probs = (np.arange(1, n + 1) - 0.5) / n
    theoretical = dist.ppf(probs)

    return theoretical, x

local_rng = np.random.default_rng(SEED + 61)

normal_sample = local_rng.normal(size=250)
heavy_sample = local_rng.standard_t(df=3, size=250)

def qq_plot(sample, title):
    q_theory, q_sample = qq_data(sample)

    lo = min(q_theory.min(), q_sample.min())
    hi = max(q_theory.max(), q_sample.max())

    source = ColumnDataSource(
        {
            "theoretical": q_theory,
            "observed": q_sample,
        }
    )

    p = figure(
        width=700,
        height=500,
        title=title,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    p.scatter(
        "theoretical",
        "observed",
        source=source,
        size=7,
        alpha=0.7,
    )

    p.line(
        [lo, hi],
        [lo, hi],
        line_dash="dashed",
        line_width=2,
    )

    p.add_tools(
        HoverTool(
            tooltips=[
                ("Theoretical", "@theoretical{0.000}"),
                ("Observed", "@observed{0.000}"),
            ]
        )
    )

    p.xaxis.axis_label = "Theoretical normal quantile"
    p.yaxis.axis_label = "Observed sample quantile"

    return p

show(
    column(
        qq_plot(
            normal_sample,
            "Normal QQ plot — data truly Normal",
        ),
        qq_plot(
            heavy_sample,
            "Normal QQ plot — heavy-tailed t(3) data",
        ),
    )
)

### Reading QQ plots

Patterns worth recognising:

- roughly straight line → family plausible;
- strong S-shape → tail mismatch;
- curvature → skewness / scale mismatch;
- isolated endpoint deviations → possible tail outliers.

QQ plots diagnose *how* a distribution differs, which is often more informative than returning a binary normality-test decision.

# Part X — Categorical and Distribution-Free Inference

## 23. Chi-square goodness-of-fit

Suppose observations fall into \(k\) categories.

Under \(H_0\), expected counts are

\[
E_i=np_i.
\]

Pearson's statistic is

\[
X^2
=
\sum_{i=1}^k
\frac{(O_i-E_i)^2}{E_i}.
\]

For sufficiently large expected counts and under \(H_0\),

\[
X^2
\approx
\chi^2_{k-1-r},
\]

where \(r\) parameters were estimated from the same data.

### Case Study 11 — Traffic-source distribution

An analytics team expects website sessions to arrive in proportions

\[
(0.40,\;0.30,\;0.20,\;0.10)
\]

from Search, Direct, Social and Referral.

Observed counts are compared with the expected distribution.

In [26]:
categories = ["Search", "Direct", "Social", "Referral"]
expected_prob = np.array([0.40, 0.30, 0.20, 0.10])
observed = np.array([455, 286, 178, 81])

n = observed.sum()
expected = n * expected_prob

chi2_stat = np.sum((observed - expected) ** 2 / expected)
df = len(categories) - 1
p_value = stats.chi2.sf(chi2_stat, df)

gof_table = pd.DataFrame(
    {
        "Category": categories,
        "Observed": observed,
        "Expected": expected,
        "Contribution": (observed - expected) ** 2 / expected,
    }
)

display(gof_table)

print(f"Chi-square statistic = {chi2_stat:.4f}")
print(f"df = {df}")
print(f"p-value = {p_value:.6f}")

,Category,Observed,Expected,Contribution
0,Search,455,400.0,7.5625
1,Direct,286,300.0,0.6533
2,Social,178,200.0,2.4200
3,Referral,81,100.0,3.6100


Chi-square statistic = 14.2458
df = 3
p-value = 0.002589


In [27]:
x = np.arange(len(categories))

source = ColumnDataSource(
    {
        "category": categories,
        "observed": observed,
        "expected": expected,
    }
)

p = figure(
    x_range=categories,
    width=900,
    height=420,
    title="Observed versus expected traffic-source counts",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.vbar(
    x="category",
    top="observed",
    width=0.6,
    source=source,
    alpha=0.55,
    legend_label="Observed",
)

p.scatter(
    x="category",
    y="expected",
    source=source,
    size=14,
    marker="diamond",
    legend_label="Expected",
)

p.add_tools(
    HoverTool(
        tooltips=[
            ("Category", "@category"),
            ("Observed", "@observed"),
            ("Expected", "@expected{0.0}"),
        ]
    )
)

p.yaxis.axis_label = "Count"
p.legend.click_policy = "hide"

show(p)

## 24. Chi-square test of independence

For a contingency table with row index \(i\) and column index \(j\),

\[
E_{ij}
=
\frac{
(\text{row total})_i
(\text{column total})_j
}{
N
}.
\]

Then

\[
X^2
=
\sum_i\sum_j
\frac{(O_{ij}-E_{ij})^2}{E_{ij}}.
\]

Degrees of freedom:

\[
(r-1)(c-1).
\]

### Case Study 12 — Device type versus purchase

We test whether purchase behaviour is independent of device type.

In [28]:
observed = np.array(
    [
        [120, 380],   # Mobile: purchased, not purchased
        [95, 205],    # Desktop
        [35, 165],    # Tablet
    ]
)

device_labels = ["Mobile", "Desktop", "Tablet"]
purchase_labels = ["Purchased", "Not purchased"]

chi2_stat, p_value, dof, expected = stats.chi2_contingency(
    observed,
    correction=False,
)

print("Observed")
display(
    pd.DataFrame(
        observed,
        index=device_labels,
        columns=purchase_labels,
    )
)

print("Expected under independence")
display(
    pd.DataFrame(
        expected,
        index=device_labels,
        columns=purchase_labels,
    )
)

print(f"Chi-square = {chi2_stat:.4f}")
print(f"df = {dof}")
print(f"p-value = {p_value:.6f}")

Observed


,Purchased,Not purchased
Mobile,120,380
Desktop,95,205
Tablet,35,165


Expected under independence


,Purchased,Not purchased
Mobile,125.0,375.0
Desktop,75.0,225.0
Tablet,50.0,150.0


Chi-square = 13.3778
df = 2
p-value = 0.001245


## 25. Distribution-free inference

Classical \(t\)-procedures rely on normal-theory assumptions for exact finite-sample results.

Distribution-free / rank-based procedures weaken assumptions.

Two useful examples are:

- **Sign test**
- **Wilcoxon signed-rank test**

---

### Sign test

For paired differences \(D_i\), under a null median of zero and continuity,

\[
P(D_i>0)=\frac12.
\]

Ignoring ties, the number of positive signs therefore follows

\[
B\sim Binomial(n,1/2).
\]

### Wilcoxon signed-rank

Wilcoxon uses both:

- sign;
- rank of absolute magnitude.

It can gain power when the difference distribution is roughly symmetric.

### Case Study 13 — Paired latency optimisation

We measure API response time on the same workloads before and after an optimisation.

The distribution contains outliers, making rank-based analysis appealing.

In [29]:
local_rng = np.random.default_rng(SEED + 70)

n = 30

baseline = local_rng.lognormal(
    mean=np.log(180),
    sigma=0.35,
    size=n,
)

improvement = local_rng.normal(
    loc=18,
    scale=10,
    size=n,
)

after = baseline - improvement

# Add a few unusual workloads.
after[[3, 17]] += np.array([60, 90])

difference = baseline - after

positive = np.sum(difference > 0)
nonzero = np.sum(difference != 0)

sign_test = stats.binomtest(
    positive,
    nonzero,
    p=0.5,
    alternative="greater",
)

wilcoxon_test = stats.wilcoxon(
    difference,
    alternative="greater",
    zero_method="wilcox",
)

paired_t = stats.ttest_rel(
    baseline,
    after,
    alternative="greater",
)

pd.DataFrame(
    {
        "Procedure": [
            "Sign test",
            "Wilcoxon signed-rank",
            "Paired t-test",
        ],
        "Statistic": [
            positive,
            wilcoxon_test.statistic,
            paired_t.statistic,
        ],
        "p-value": [
            sign_test.pvalue,
            wilcoxon_test.pvalue,
            paired_t.pvalue,
        ],
    }
)

,Procedure,Statistic,p-value
0,Sign test,27.0000,4.2152e-06
1,Wilcoxon signed-rank,408.0000,6.1668e-05
2,Paired t-test,2.9415,3.1803e-03


In [30]:
source = ColumnDataSource(
    {
        "workload": np.arange(1, n + 1),
        "difference": difference,
    }
)

p = figure(
    width=900,
    height=400,
    title="Paired latency improvement: baseline − after",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.scatter(
    "workload",
    "difference",
    source=source,
    size=9,
    alpha=0.7,
)

p.add_layout(
    Span(
        location=0,
        dimension="width",
        line_dash="dashed",
        line_width=2,
    )
)

p.xaxis.axis_label = "Workload"
p.yaxis.axis_label = "Latency reduction"

p.add_tools(
    HoverTool(
        tooltips=[
            ("Workload", "@workload"),
            ("Difference", "@difference{0.00}"),
        ]
    )
)

show(p)

# Part XI — Optional Bridge to Regression Inference

## 26. Simple linear regression as likelihood-based inference

Consider

\[
Y_i=\beta_0+\beta_1x_i+\epsilon_i,
\qquad
\epsilon_i\overset{iid}{\sim}N(0,\sigma^2).
\]

The log-likelihood is, up to constants,

\[
\ell(\beta_0,\beta_1)
=
-\frac{1}{2\sigma^2}
\sum_i
(y_i-\beta_0-\beta_1x_i)^2.
\]

Hence maximising likelihood is equivalent to minimising residual sum of squares.

This creates a direct bridge:

\[
\boxed{
\text{Gaussian likelihood}
\Longleftrightarrow
\text{ordinary least squares}
}
\]

Inference then asks:

- Is \(\hat\beta_1\) unbiased?
- What is \(SE(\hat\beta_1)\)?
- What is a confidence interval for \(\beta_1\)?
- Can we test \(H_0:\beta_1=0\)?

In [31]:
local_rng = np.random.default_rng(SEED + 80)

n = 80
x = np.linspace(0, 10, n)

beta0 = 3.0
beta1 = 1.8
sigma = 3.0

y = beta0 + beta1 * x + local_rng.normal(
    0,
    sigma,
    size=n,
)

x_bar = x.mean()
y_bar = y.mean()

Sxx = np.sum((x - x_bar) ** 2)
Sxy = np.sum((x - x_bar) * (y - y_bar))

b1 = Sxy / Sxx
b0 = y_bar - b1 * x_bar

resid = y - (b0 + b1 * x)
s2 = np.sum(resid**2) / (n - 2)
se_b1 = np.sqrt(s2 / Sxx)

t_stat = b1 / se_b1
p_value = 2 * stats.t.sf(abs(t_stat), df=n - 2)

tcrit = stats.t.ppf(0.975, df=n - 2)
ci = (
    b1 - tcrit * se_b1,
    b1 + tcrit * se_b1,
)

print(f"Estimated intercept = {b0:.4f}")
print(f"Estimated slope     = {b1:.4f}")
print(f"SE(slope)           = {se_b1:.4f}")
print(f"t statistic         = {t_stat:.4f}")
print(f"p-value             = {p_value:.8f}")
print(f"95% CI for slope    = ({ci[0]:.4f}, {ci[1]:.4f})")

Estimated intercept = 3.1730
Estimated slope     = 1.8738
SE(slope)           = 0.1285
t statistic         = 14.5815
p-value             = 0.00000000
95% CI for slope    = (1.6179, 2.1296)


In [32]:
x_line = np.linspace(x.min(), x.max(), 300)
y_line = b0 + b1 * x_line

source = ColumnDataSource({"x": x, "y": y})

p = figure(
    width=900,
    height=420,
    title="Simple linear regression: estimated mean relationship",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.scatter(
    "x",
    "y",
    source=source,
    size=8,
    alpha=0.65,
    legend_label="Observed data",
)

p.line(
    x_line,
    y_line,
    line_width=3,
    legend_label="Fitted line",
)

p.xaxis.axis_label = "x"
p.yaxis.axis_label = "y"
p.legend.click_policy = "hide"

show(p)

# Part XII — Integrated Case Study

## 27. End-to-end case: transaction anomaly rate

A fraud-monitoring platform has historically flagged 2% of transactions.

After a rules-engine update, 4,000 transactions produce 105 alerts.

We will walk through the ST2132 pipeline:

\[
\text{Model}
\rightarrow
\text{Estimator}
\rightarrow
\text{Standard error}
\rightarrow
\text{Confidence interval}
\rightarrow
\text{Hypothesis test}
\rightarrow
\text{Likelihood}
\rightarrow
\text{Power}
\]

Assume

\[
X_i\sim Bernoulli(p).
\]

In [33]:
n = 4_000
x = 105
p0 = 0.02

p_hat = x / n

# Approximate standard error evaluated at MLE.
se_hat = np.sqrt(p_hat * (1 - p_hat) / n)

# Wald confidence interval.
zcrit = stats.norm.ppf(0.975)

wald_ci = (
    p_hat - zcrit * se_hat,
    p_hat + zcrit * se_hat,
)

# Score-style null test.
z_stat = (
    (p_hat - p0)
    / np.sqrt(p0 * (1 - p0) / n)
)

p_value = 2 * stats.norm.sf(abs(z_stat))

summary = pd.Series(
    {
        "n": n,
        "alerts": x,
        "MLE p_hat": p_hat,
        "SE(p_hat)": se_hat,
        "95% Wald lower": wald_ci[0],
        "95% Wald upper": wald_ci[1],
        "Z statistic for H0:p=0.02": z_stat,
        "Two-sided p-value": p_value,
    }
)

summary

n                            4000.0000
alerts                        105.0000
MLE p_hat                       0.0262
SE(p_hat)                       0.0025
95% Wald lower                  0.0213
95% Wald upper                  0.0312
Z statistic for H0:p=0.02       2.8235
Two-sided p-value               0.0048
dtype: float64

In [34]:
p_grid = np.linspace(0.008, 0.045, 700)

log_likelihood = (
    x * np.log(p_grid)
    + (n - x) * np.log(1 - p_grid)
)

log_likelihood -= log_likelihood.max()

fig = figure(
    width=900,
    height=420,
    title="Bernoulli likelihood for post-update anomaly rate",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

fig.line(
    p_grid,
    log_likelihood,
    line_width=3,
    legend_label="Relative log-likelihood",
)

fig.add_layout(
    Span(
        location=p_hat,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

fig.add_layout(
    Span(
        location=p0,
        dimension="height",
        line_dash="dotted",
        line_width=2,
    )
)

fig.xaxis.axis_label = "Candidate anomaly probability p"
fig.yaxis.axis_label = "Relative log-likelihood"
fig.legend.click_policy = "hide"

show(fig)

In [35]:
alpha = 0.05

# Two-sided approximate score test: rejection if |Z| > z_(1-alpha/2).
zcrit = stats.norm.ppf(1 - alpha / 2)

lower_threshold = (
    p0
    - zcrit * np.sqrt(p0 * (1 - p0) / n)
)

upper_threshold = (
    p0
    + zcrit * np.sqrt(p0 * (1 - p0) / n)
)

true_p_grid = np.linspace(0.005, 0.05, 600)

sd_alt = np.sqrt(
    true_p_grid * (1 - true_p_grid) / n
)

power = (
    stats.norm.cdf(
        (lower_threshold - true_p_grid) / sd_alt
    )
    +
    1
    -
    stats.norm.cdf(
        (upper_threshold - true_p_grid) / sd_alt
    )
)

fig = figure(
    width=900,
    height=420,
    title="Approximate power curve for detecting anomaly-rate changes",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

fig.line(
    true_p_grid,
    power,
    line_width=3,
)

fig.add_layout(
    Span(
        location=p0,
        dimension="height",
        line_dash="dashed",
        line_width=2,
    )
)

fig.xaxis.axis_label = "True anomaly probability p"
fig.yaxis.axis_label = "Power"

show(fig)

print(f"Approx. power if true p=0.025: {np.interp(0.025, true_p_grid, power):.3f}")
print(f"Approx. power if true p=0.030: {np.interp(0.030, true_p_grid, power):.3f}")

Approx. power if true p=0.025: 0.606
Approx. power if true p=0.030: 0.982


### Integrated interpretation

The case study links several ST2132 concepts:

1. **Sampling model**
   \[
   X\sim Binomial(n,p)
   \]

2. **MLE**
   \[
   \hat p=\frac{X}{n}
   \]

3. **Estimator variance**
   \[
   Var(\hat p)=\frac{p(1-p)}{n}
   \]

4. **Fisher information**
   \[
   I_n(p)=\frac{n}{p(1-p)}
   \]

5. **CRLB**
   \[
   \frac{1}{I_n(p)}
   =
   \frac{p(1-p)}{n}
   \]

6. **Approximate confidence interval**
   built from asymptotic normality.

7. **Hypothesis test**
   compares observed deviation with null sampling variability.

8. **Power**
   quantifies how likely the procedure is to detect a genuine departure from \(p_0\).

This is the core workflow of mathematical statistics:

\[
\boxed{
\text{probability model}
\rightarrow
\text{sampling distribution}
\rightarrow
\text{estimation}
\rightarrow
\text{uncertainty}
\rightarrow
\text{decision}
}
\]

# Part XIII — Interactive Bokeh Experiment

## 28. Interactive Normal likelihood curvature

Use the slider to change the effective sample size \(n\).

For known \(\sigma\),

\[
\ell(\mu)-\ell(\hat\mu)
=
-\frac{n}{2\sigma^2}(\mu-\hat\mu)^2.
\]

This isolates the relationship between:

\[
n
\longrightarrow
\text{likelihood curvature}
\longrightarrow
\text{Fisher information}
\longrightarrow
\text{estimator precision}.
\]

In [36]:
mu_grid = np.linspace(-2.5, 2.5, 600)
sigma = 1.0

n_initial = 10
relative_ll = -n_initial * mu_grid**2 / (2 * sigma**2)

source = ColumnDataSource(
    {
        "mu": mu_grid,
        "ll": relative_ll,
    }
)

p = figure(
    width=900,
    height=430,
    title="Interactive likelihood curvature",
    y_range=(-30, 1),
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.line(
    "mu",
    "ll",
    source=source,
    line_width=3,
)

p.xaxis.axis_label = "μ − μ̂"
p.yaxis.axis_label = "Relative log-likelihood"

slider = Slider(
    start=1,
    end=100,
    value=n_initial,
    step=1,
    title="Sample size n",
)

callback = CustomJS(
    args=dict(
        source=source,
        slider=slider,
        sigma=sigma,
    ),
    code="""
        const data = source.data;
        const mu = data['mu'];
        const ll = data['ll'];
        const n = slider.value;

        for (let i = 0; i < mu.length; i++) {
            ll[i] = -n * mu[i] * mu[i] / (2 * sigma * sigma);
        }

        source.change.emit();
    """,
)

slider.js_on_change("value", callback)

show(column(slider, p))

### Interpretation

As \(n\) increases:

\[
I_n(\mu)=\frac{n}{\sigma^2}
\]

increases linearly.

At the same time the likelihood becomes sharper, and

\[
SE(\hat\mu)
=
\frac{\sigma}{\sqrt n}
\]

shrinks.

This visual ties together three apparently different concepts:

- likelihood geometry,
- Fisher information,
- standard error.

They are different views of the same inferential phenomenon.

# Part XIV — Exercises

## 29. Exercises

### Exercise A — MLE for a Poisson model

Suppose

\[
X_i\sim Poisson(\lambda).
\]

1. Write the likelihood.
2. Write the log-likelihood.
3. Differentiate it.
4. Show that
   \[
   \hat\lambda=\bar X.
   \]
5. Simulate the estimator for \(n\in\{5,20,100,500\}\).
6. Use Bokeh to visualise how its sampling distribution contracts.

---

### Exercise B — Fisher information for Poisson

Show that for one observation,

\[
I_1(\lambda)=\frac{1}{\lambda},
\]

and therefore

\[
I_n(\lambda)=\frac{n}{\lambda}.
\]

Compare

\[
1/I_n(\lambda)
\]

with the variance of \(\bar X\).

---

### Exercise C — Confidence-interval coverage

For an exponential population:

1. build the naive normal CLT interval for the mean;
2. estimate empirical coverage at \(n=5,20,50,200\);
3. determine how skewness affects small-sample coverage.

---

### Exercise D — Type-I error and power

For

\[
H_0:\mu=0
\quad\text{vs}\quad
H_1:\mu>0
\]

with known \(\sigma=1\):

1. construct a size 0.05 test;
2. simulate its rejection rate under \(\mu=0\);
3. simulate power for \(\mu\in[0,1]\);
4. repeat for \(n=10,25,100\).

---

### Exercise E — Order statistics

For

\[
X_i\sim Uniform(0,1),
\]

simulate \(X_{(1)}\), \(X_{(5)}\), and \(X_{(10)}\) for \(n=10\).

Overlay the corresponding theoretical Beta densities.

---

### Exercise F — QQ plot diagnostics

Generate datasets from:

- Normal,
- Student-\(t_3\),
- Exponential,
- Lognormal.

Create normal QQ plots and describe the distinctive curvature of each.

# Part XV — Selected Exercise Solutions

## 30. Solution A — Poisson MLE

For

\[
X_i\sim Poisson(\lambda),
\]

\[
f(x_i;\lambda)
=
e^{-\lambda}\frac{\lambda^{x_i}}{x_i!}.
\]

Therefore

\[
L(\lambda)
=
e^{-n\lambda}
\frac{
\lambda^{\sum_i x_i}
}{
\prod_i x_i!
}.
\]

The log-likelihood is

\[
\ell(\lambda)
=
-n\lambda
+
\left(\sum_i x_i\right)\log\lambda
-
\sum_i\log(x_i!).
\]

Score:

\[
U(\lambda)
=
-n+\frac{\sum_i x_i}{\lambda}.
\]

Set \(U(\lambda)=0\):

\[
\hat\lambda
=
\frac{1}{n}\sum_i x_i
=
\bar X.
\]

In [37]:
local_rng = np.random.default_rng(SEED + 100)

true_lambda = 4.0

for n in [5, 20, 100, 500]:
    estimates = local_rng.poisson(
        true_lambda,
        size=(20_000, n)
    ).mean(axis=1)

    p = histogram_density_plot(
        estimates,
        title=f"Poisson MLE sampling distribution — n={n}",
        x_label="λ-hat = X̄",
        bins=55,
    )

    p.add_layout(
        Span(
            location=true_lambda,
            dimension="height",
            line_dash="dashed",
            line_width=2,
        )
    )

    show(p)

    print(
        f"n={n:>3}: "
        f"mean={estimates.mean():.4f}, "
        f"variance={estimates.var():.5f}, "
        f"theory={true_lambda/n:.5f}"
    )

n=  5: mean=3.9978, variance=0.80613, theory=0.80000


n= 20: mean=3.9954, variance=0.19939, theory=0.20000


n=100: mean=3.9990, variance=0.03917, theory=0.04000


n=500: mean=4.0001, variance=0.00804, theory=0.00800


## 31. Solution B — Poisson Fisher information

For one observation,

\[
\ell(\lambda)
=
-\lambda+X\log\lambda-\log(X!).
\]

Differentiate twice:

\[
\frac{\partial\ell}{\partial\lambda}
=
-1+\frac{X}{\lambda},
\]

\[
\frac{\partial^2\ell}{\partial\lambda^2}
=
-\frac{X}{\lambda^2}.
\]

Therefore,

\[
I_1(\lambda)
=
-E\left[
-\frac{X}{\lambda^2}
\right]
=
\frac{E[X]}{\lambda^2}
=
\frac{\lambda}{\lambda^2}
=
\frac{1}{\lambda}.
\]

Hence

\[
I_n(\lambda)
=
\frac{n}{\lambda}
\]

and the CRLB is

\[
\frac{1}{I_n(\lambda)}
=
\frac{\lambda}{n}.
\]

But

\[
Var(\bar X)
=
\frac{Var(X)}{n}
=
\frac{\lambda}{n}.
\]

So the Poisson MLE \(\bar X\) attains the Cramér–Rao bound.

# Part XVI — Exam / Revision Map

## 32. What to be able to derive

For ST2132-style problem solving, do not stop at memorising formulas.

You should be able to derive or reconstruct:

### Sampling distributions

\[
E[\bar X],
\qquad
Var(\bar X),
\qquad
\frac{(n-1)S^2}{\sigma^2},
\qquad
\frac{\bar X-\mu}{S/\sqrt n}.
\]

### Estimation

\[
L(\theta),
\quad
\ell(\theta),
\quad
U(\theta),
\quad
\hat\theta_{\text{MLE}}.
\]

### Estimator properties

\[
Bias(\hat\theta),
\quad
Var(\hat\theta),
\quad
MSE(\hat\theta).
\]

Know the decomposition

\[
MSE=Variance+Bias^2.
\]

### Fisher information

\[
I(\theta)
=
E[U(\theta)^2]
=
-E[\ell''(\theta)].
\]

### CRLB

\[
Var(\hat\theta)
\ge
1/I_n(\theta).
\]

### Sufficiency

Factor the likelihood into

\[
g(T(x),\theta)h(x).
\]

### Confidence intervals

Find a pivot and algebraically invert the probability statement.

### Hypothesis tests

Be comfortable moving among:

\[
H_0/H_1
\rightarrow
T(X)
\rightarrow
\text{null distribution}
\rightarrow
\text{critical region}
\rightarrow
p\text{-value}
\rightarrow
\text{power}.
\]

### Neyman–Pearson

For simple-vs-simple hypotheses, compare likelihoods.

### LRT

\[
\Lambda
=
\frac{
\sup_{\Theta_0}L(\theta)
}{
\sup_{\Theta}L(\theta)
}.
\]

Understand why small \(\Lambda\) is evidence against \(H_0\).

### Order statistics

Know how to derive the CDF/density of \(X_{(k)}\), and remember the Uniform-to-Beta relationship.

---

## Final mental model

ST2132 is best viewed as:

\[
\boxed{
\begin{array}{c}
\text{Probability model}\\
\downarrow\\
\text{Random sample}\\
\downarrow\\
\text{Statistic / estimator}\\
\downarrow\\
\text{Sampling distribution}\\
\downarrow\\
\text{Estimator quality}\\
\downarrow\\
\text{Confidence interval / test}\\
\downarrow\\
\text{Decision with quantified uncertainty}
\end{array}
}
\]

The common theme is not “calculate a p-value”.

It is:

> **Understand how random data induce random estimators and decisions, then quantify their behaviour mathematically.**

## 33. Suggested extensions

Once you are comfortable with this notebook, natural follow-ons are:

- bootstrap confidence intervals;
- delta method;
- exponential families;
- Rao–Blackwellisation;
- Bayesian estimators versus frequentist estimators;
- Wald, score and likelihood-ratio test comparison;
- asymptotic normality of MLEs;
- profile likelihood;
- multiple testing;
- regression and generalized linear models.

Those topics show how the ST2132 foundation scales into modern statistical modelling and machine learning.